<a href="https://colab.research.google.com/github/24nga/datafiles/blob/%EB%8D%B0%EC%9D%B4%ED%84%B0-%EB%B6%84%EC%84%9D/2025_%EB%85%BC%EB%AC%B8_BertScore%2BSBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SBERT 코사인: 문장 전체를 임베딩으로 만든 뒤 의미 거리(방향)를 비교 → 요구사항 문장 “의미 유사도”에 직관적.

BERTScore(F1): 토큰 단위로 contextual embedding 정렬 후 유사도 계산 → 어휘가 달라도 의미가 맞으면 점수가 잘 나오는 편(코사인보다 정밀한 경우가 많음).

| dataset        |   n |   mean | median |    p10 |    p90 |
| -------------- | --: | -----: | -----: | -----: | -----: |
| chatgpt_think  | 166 | 0.2869 | 0.2945 | 0.0970 | 0.4531 |
| chatgpt_fast   | 275 | 0.2301 | 0.2296 | 0.0616 | 0.4026 |
| deepseek_r1_8b |  45 | 0.1754 | 0.1228 | 0.0044 | 0.4103 |
| gemma3_12b     | 164 | 0.1633 | 0.1411 | 0.0263 | 0.3339 |
| gemini_think   | 185 | 0.0803 | 0.0729 | 0.0257 | 0.1439 |
| gemini_fast    | 179 | 0.0475 | 0.0163 | 0.0014 | 0.1340 |




1) 설치


In [1]:
!pip -q install sentence-transformers bert-score openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 838.2 kB/s eta 0:00:00


2) 엑셀 로드

In [2]:
import pandas as pd
import numpy as np

in_path = "/content/sample_similarity_results_recalc.xlsx"  # 코랩에 업로드한 파일명으로 맞추세요
df_all = pd.read_excel(in_path, sheet_name="all_rows")

# 비교 텍스트 (이미 best match가 잡혀있는 구조)
cands = df_all["gen_text"].fillna("").astype(str).tolist()
refs  = df_all["best_ref_text"].fillna("").astype(str).tolist()

# 빈 문자열만 있는 행 제외(선택)
mask = [(len(c.strip()) > 0 and len(r.strip()) > 0) for c, r in zip(cands, refs)]
valid_idx = np.where(mask)[0]


3) SBERT 코사인 (권장: 다국어 MiniLM)

In [3]:
from sentence_transformers import SentenceTransformer, util
import torch

# 다국어(한국어 포함) 범용 모델
sbert_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(sbert_name)

# 임베딩
cand_emb = model.encode([cands[i] for i in valid_idx], batch_size=64, convert_to_tensor=True, show_progress_bar=True)
ref_emb  = model.encode([refs[i]  for i in valid_idx], batch_size=64, convert_to_tensor=True, show_progress_bar=True)

# 행별 1:1 코사인(대각선)
cos = util.cos_sim(cand_emb, ref_emb).diagonal().detach().cpu().numpy()

df_all["sbert_cosine"] = np.nan
df_all.loc[valid_idx, "sbert_cosine"] = cos
df_all["sbert_cosine"].describe()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

,sbert_cosine
count,1014.000000
mean,0.573209
std,0.163973
min,0.000209
25%,0.485960
50%,0.594989
75%,0.687446
max,0.958632


4) BERTScore(F1) (권장: XLM-RoBERTa large)

In [5]:
from bert_score import score as bertscore

# GPU 있으면 자동 사용
device = "cuda" if torch.cuda.is_available() else "cpu"

P, R, F1 = bertscore(
    cands=[cands[i] for i in valid_idx],
    refs=[refs[i]  for i in valid_idx],
    model_type="xlm-roberta-large",  # 한국어 포함 다국어에 무난
    lang="ko",
    device=device,
    batch_size=16,
    verbose=True
)

df_all["bertscore_f1"] = np.nan
df_all.loc[valid_idx, "bertscore_f1"] = F1.detach().cpu().numpy()
df_all["bertscore_f1"].describe()


calculating scores...
computing bert embedding.


  0%|          | 0/65 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/64 [00:00<?, ?it/s]

done in 251.42 seconds, 4.03 sentences/sec


,bertscore_f1
count,1014.000000
mean,0.871617
std,0.027676
min,0.734938
25%,0.857714
50%,0.873080
75%,0.888600
max,0.954073


5) 요약 통계 생성 + 저장


In [7]:
def summarize(df, metric_col):
    g = df.groupby("dataset")[metric_col]
    return pd.DataFrame({
        "dataset": g.count().index,
        "n": g.count().values,
        f"{metric_col}_mean": g.mean().values,
        f"{metric_col}_median": g.median().values,
        f"{metric_col}_p10": g.quantile(0.1).values,
        f"{metric_col}_p90": g.quantile(0.9).values,
    })

sum_tfidf = summarize(df_all, "cosine_tfidf")
sum_sbert = summarize(df_all, "sbert_cosine")
sum_bert  = summarize(df_all, "bertscore_f1")

# 하나로 합치기
summary = sum_tfidf.merge(sum_sbert, on=["dataset","n"], how="outer").merge(sum_bert, on=["dataset","n"], how="outer")

out_path = "/content/similarity_results_with_sbert_bertscore.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary.to_excel(w, index=False, sheet_name="summary_all")
    df_all.to_excel(w, index=False, sheet_name="all_rows")

out_path


'/content/similarity_results_with_sbert_bertscore.xlsx'